In [1]:
# Phase 1 (Transfer Learning): Model initialization only
# - Backbone: MobileNetV2 (ImageNet pretrained)
# - Classes: happy / neutral / sad (3 classes)
# - Input: 224x224 RGB
#
# Academic note: This phase intentionally excludes any dataset loading/training.

import tensorflow as tf
from tensorflow.keras import layers


/Users/mariam/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mariam/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# 1) Load MobileNetV2 backbone (ImageNet pretrained)
#    - include_top=False removes the 1000-class ImageNet classifier head
#    - alpha=1.0 keeps the standard MobileNetV2 width multiplier
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    alpha=1.0,
    input_shape=(224, 224, 3),
)

In [4]:
# 2) Freeze the entire backbone (transfer learning Phase 1 requirement)
base_model.trainable = False

In [5]:
# 3) Build the new classification head for 3 emotion classes
#    Head: GlobalAveragePooling2D -> Dropout(0.3) -> Dense(3, softmax)
inputs = tf.keras.Input(shape=(224, 224, 3), name="image")

# MobileNetV2 expects inputs preprocessed this way
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

# Forward pass through frozen backbone
x = base_model(x, training=False)

# Custom head
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="dropout_0_3")(x)
outputs = layers.Dense(3, activation="softmax", name="emotion_logits")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="emotion_mobilenetv2_phase1")

In [6]:
# 4) Compile (Phase 1 requirement)
# - Optimizer: Adam
# - Learning rate: 0.001
# - Loss: sparse_categorical_crossentropy (expects integer class labels: 0/1/2)
# - Metric: accuracy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [7]:
# 5) Inspect architecture (required)
model.summary()

# 6) Save the Phase 1 model (required)
# This saves the full Keras model (architecture + weights).
model.save("emotion_model_phase1.h5")
print("Saved model to emotion_model_phase1.h5")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Phase 2 (later)

Dataset loading, preprocessing, and fine-tuning will be added in a later phase.

**Per Phase 1 requirements, no dataset code is included here.**

DatasetNotFoundError: Dataset fer2013 not found.
Available datasets:
	- abstract_reasoning
	- accentdb
	- aeslc
	- aflw2k3d
	- ag_news_subset
	- ai2_arc
	- ai2_arc_with_ir
	- amazon_us_reviews
	- anli
	- answer_equivalence
	- arc
	- asqa
	- asset
	- assin2
	- bair_robot_pushing_small
	- bccd
	- beans
	- bee_dataset
	- beir
	- big_patent
	- bigearthnet
	- billsum
	- binarized_mnist
	- binary_alpha_digits
	- ble_wind_field
	- blimp
	- booksum
	- bool_q
	- bot_adversarial_dialogue
	- bucc
	- c4
	- c4_wsrs
	- caltech101
	- caltech_birds2010
	- caltech_birds2011
	- cardiotox
	- cars196
	- cassava
	- cats_vs_dogs
	- celeb_a
	- celeb_a_hq
	- cfq
	- cherry_blossoms
	- chexpert
	- cifar10
	- cifar100
	- cifar100_n
	- cifar10_1
	- cifar10_corrupted
	- cifar10_h
	- cifar10_n
	- citrus_leaves
	- cityscapes
	- civil_comments
	- clevr
	- clic
	- clinc_oos
	- cmaterdb
	- cnn_dailymail
	- coco
	- coco_captions
	- coil100
	- colorectal_histology
	- colorectal_histology_large
	- common_voice
	- conll200

NameError: name 'train_dataset' is not defined